# 03 什么是 Gradient Noise Scale，如何用它指导 batch size？

## 面试回答主线

Gradient Noise Scale（GNS）把样本梯度的随机方差与平均梯度信号作比，刻画增大 batch 后还能减少多少优化噪声。直觉上，若样本梯度彼此方向分散，较大的 batch 仍有价值；若平均梯度已经很强，继续堆 batch 往往只减少更新次数。面试中要说它是在线诊断量，不是固定常数，也不能单独决定显存、吞吐和泛化。实验对 6 条人工客服工单逐样本计算逻辑回归梯度，展示 GNS、建议 batch 和“sum 代替 mean”的估计错误。教学数据不能外推为生产集群的临界 batch。

**核心公式：** 令单样本梯度为 $g_i$、均值为 $\bar g$，可用 $B_{noise}\approx\operatorname{tr}(\operatorname{Cov}(g_i))/(\lVert\bar g\rVert_2^2+\epsilon)$ 估计噪声尺度；它与有用 batch 的量级相关而非严格相等。

本 Notebook 将依次展示业务输入、可比较基线、手写核心机制、中间量、失败与修复；所有数值都是确定性的教学实验。


## 真实案例

场景是支付与账户安全客服系统：模型要把工单分成“高风险需优先人工处理”和“常规处理”。三个输入特征分别表示资金风险线索、登录/身份线索和售后/账单线索。数据为人工构造的脱敏离线事件，字段结构模拟真实工单，不可外推为生产表现。


In [1]:
import math  # 导入数学函数以实现尺度公式。
import warnings  # 导入警告控制模块以保持教学输出干净。
warnings.filterwarnings('ignore', message='The pynvml package is deprecated')  # 屏蔽环境依赖产生的非教学弃用警告。
import torch  # 导入 PyTorch 张量和自动微分能力。
import torch.nn as nn  # 导入模块基类以手写网络结构。
torch.manual_seed(17)  # 固定随机种子使教学输出可复现。
torch.set_num_threads(1)  # 限制 CPU 线程以减少小实验波动。
samples = [  # 构造脱敏客服工单的真实语义样本。
    {'ticket': '支付重复扣款，要求退款', 'features': [1.0, 0.0, 1.0], 'label': 1},  # 高风险退款工单。
    {'ticket': '登录验证码收不到', 'features': [0.0, 1.0, 0.0], 'label': 0},  # 普通技术支持工单。
    {'ticket': '账户出现陌生转账', 'features': [1.0, 0.0, 0.0], 'label': 1},  # 高风险资金安全工单。
    {'ticket': '如何修改收货地址', 'features': [0.0, 0.0, 1.0], 'label': 0},  # 普通售后咨询工单。
    {'ticket': '银行卡被盗刷请冻结', 'features': [1.0, 1.0, 0.0], 'label': 1},  # 高风险且紧急的工单。
    {'ticket': '发票抬头需要更正', 'features': [0.0, 1.0, 1.0], 'label': 0},  # 低风险但需要人工处理的工单。
]  # 结束教学样本定义。
features = torch.tensor([row['features'] for row in samples], dtype=torch.float32)  # 将可读字段转为模型输入张量。
labels = torch.tensor([row['label'] for row in samples], dtype=torch.long)  # 将风险标签转为分类目标。
print('教学实验：脱敏客服工单，不代表线上规模或泛化收益。')  # 明确实验边界。
for row in samples:  # 逐条展示输入样本而不是隐藏在张量中。
    print(f"标签={row['label']} | 特征={row['features']} | 工单={row['ticket']}")  # 输出原始业务语义。
print(f'输入张量形状={tuple(features.shape)}，标签={labels.tolist()}')  # 输出张量形状和目标。


教学实验：脱敏客服工单，不代表线上规模或泛化收益。
标签=1 | 特征=[1.0, 0.0, 1.0] | 工单=支付重复扣款，要求退款
标签=0 | 特征=[0.0, 1.0, 0.0] | 工单=登录验证码收不到
标签=1 | 特征=[1.0, 0.0, 0.0] | 工单=账户出现陌生转账
标签=0 | 特征=[0.0, 0.0, 1.0] | 工单=如何修改收货地址
标签=1 | 特征=[1.0, 1.0, 0.0] | 工单=银行卡被盗刷请冻结
标签=0 | 特征=[0.0, 1.0, 1.0] | 工单=发票抬头需要更正
输入张量形状=(6, 3)，标签=[1, 0, 1, 0, 1, 0]


## Baseline / 基线

先看最简单的对照。基线与核心方案使用完全相同的样本、标签和指标，避免把数据变化误认为算法收益。


In [2]:
weight = torch.tensor([0.2, -0.1, 0.3])  # 设置当前逻辑回归权重模拟训练中间态。
batch_logit = features @ weight  # 计算全 batch logits。
batch_prob = torch.sigmoid(batch_logit)  # 将 logits 转为正类概率。
batch_grad = (features.T @ (batch_prob - labels.float())) / len(samples)  # 计算平均 batch 梯度作为基线。
baseline_metric = float(batch_grad.norm())  # 保存平均梯度范数。
print(f'平均 batch 梯度={batch_grad.tolist()}，范数={baseline_metric:.4f}')  # 展示基线梯度而非只输出布尔值。


平均 batch 梯度=[-0.21712124347686768, 0.09163900464773178, 0.12445598095655441]，范数=0.2665


## 手写核心实现与中间量

以下实现刻意保留关键矩阵、梯度、范数或调度状态，目的是让面试时能解释“它到底改变了哪一个量”。


In [3]:
per_example_grads = []  # 建立逐样本梯度列表。
for row, target in zip(features, labels.float()):  # 遍历每条有语义的客服工单特征。
    probability = torch.sigmoid(row @ weight)  # 计算当前工单被判为高风险的概率。
    per_example_grads.append((probability - target) * row)  # 根据逻辑回归公式手写单样本梯度。
gradient_matrix = torch.stack(per_example_grads)  # 将逐样本梯度堆成可统计矩阵。
mean_gradient = gradient_matrix.mean(dim=0)  # 用均值保留 batch 不变的信号尺度。
centered = gradient_matrix - mean_gradient  # 去均值得到噪声部分。
trace_covariance = float(centered.pow(2).sum(dim=1).mean())  # 估计梯度协方差迹。
signal_power = float(mean_gradient.pow(2).sum())  # 计算平均梯度的信号能量。
noise_scale = trace_covariance / (signal_power + 1e-8)  # 计算 Gradient Noise Scale 近似量。
suggested_batch = min(len(samples), max(1, math.ceil(noise_scale)))  # 将估计量转为教学用建议 batch。
core_metric = noise_scale  # 保存核心指标供最终检查。
print(f'逐样本梯度矩阵形状={tuple(gradient_matrix.shape)}，协方差迹={trace_covariance:.4f}，GNS={noise_scale:.3f}，建议 batch≈{suggested_batch}')  # 展示中间量和决策。


逐样本梯度矩阵形状=(6, 3)，协方差迹=0.2789，GNS=3.926，建议 batch≈4


In [4]:
comparison_rows = [('Baseline', float(baseline_metric)), ('核心机制', float(core_metric))]  # 汇总同一指标口径下的可读结果表。
for name, metric in comparison_rows:  # 逐行输出基线与核心方案。
    print(f'{name:<8} | 指标={metric:.6f}')  # 展示结果表而不是只保留变量名。


Baseline | 指标=0.266512
核心机制     | 指标=3.925892


## 结果解读

请把基线和核心输出看成机制证据而非榜单。这里的指标只在同一受控工单集上可比：核心方案展示了 **Gradient Noise Scale** 的关键状态与更新路径。真实训练需分层抽样、跨 rank 聚合和滑动窗口平滑；逐样本梯度成本高，常用 microbatch 或近似估计。

## 失败案例

接下来故意破坏一个必要条件，再用明确的门禁、尺度或统计口径修复它。这样可以避免“代码能跑”却不知道为什么线上会失效。


In [5]:
wrong_mean = gradient_matrix.sum(dim=0)  # 故意把求和误当作平均梯度。
wrong_signal = float(wrong_mean.pow(2).sum())  # 计算被人为放大的信号能量。
failure_metric = trace_covariance / (wrong_signal + 1e-8)  # 得到虚假的低噪声尺度。
correct_mean = gradient_matrix.mean(dim=0)  # 恢复按样本平均的梯度定义。
correct_signal = float(correct_mean.pow(2).sum())  # 重新计算正确信号能量。
fix_metric = trace_covariance / (correct_signal + 1e-8)  # 得到修复后的 GNS。
print(f'失败：sum 估计 GNS={failure_metric:.4f}；修复：mean 估计 GNS={fix_metric:.4f}')  # 展示统计口径错误。


失败：sum 估计 GNS=0.1091；修复：mean 估计 GNS=3.9259


## 工程取舍、常见坑与延伸追问

**工程取舍：** 真实训练需分层抽样、跨 rank 聚合和滑动窗口平滑；逐样本梯度成本高，常用 microbatch 或近似估计。

**常见坑：** 把 sum gradient 当作 mean gradient 会让分母随 batch 人为放大，得到虚假的低噪声尺度。

**延伸追问：** 序列长度不一时按样本、按 token 还是按 loss token 估计？数据并行下如何避免各 rank 的 GNS 只看到局部数据？

## 生产差距

本实验只有 6 条脱敏离线工单、CPU 和 FP32，省略了大规模 token packing、数据并行、混合精度、checkpoint、指标告警和灰度回滚。生产实现应替换为真实数据管道与观测系统，并用验证集和线上安全指标决定是否发布。


In [6]:
assert gradient_matrix.shape[0] == len(samples)  # 验证每条样本都贡献了一个梯度。
assert core_metric > 0.0  # 验证当前批次存在可测量的梯度噪声。
assert fix_metric > failure_metric  # 验证 sum 会虚假压低噪声尺度。
assert 1 <= suggested_batch <= len(samples)  # 验证建议 batch 不超过教学数据量。
